# BiT-MAML LOPO-CV (Paper-Parity) + Android Export\n\nThis notebook targets parity with Scientific Reports 2025 (s41598-025-13491-5) while keeping your dataset-cleaning flow intact.

In [ ]:
!pip -q install pandas numpy scikit-learn torch onnx onnxruntime onnxscript\n\nimport copy\nimport json\nimport pickle\nimport re\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.preprocessing import MinMaxScaler\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error\n\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader, TensorDataset\n\nimport onnx\nimport onnxruntime as ort

In [ ]:
# ==== Paper-parity config ====\nIN_COLAB = True\ntry:\n    from google.colab import drive\nexcept ImportError:\n    IN_COLAB = False\n\nif IN_COLAB:\n    drive.mount('/content/drive', force_remount=False)\n    PROJECT_ROOT = Path('/content')\nelse:\n    PROJECT_ROOT = Path.cwd()\n\nDATASETS_DIR = PROJECT_ROOT / 'drive/MyDrive/Dataset/n=183 OpenAPS Data Commons 2022 UNZIPPED'\nPROCESSED_DIR = PROJECT_ROOT / 'Processed'\nRUNTIME_DIR = PROCESSED_DIR / 'runtime'\nPROCESSED_DIR.mkdir(parents=True, exist_ok=True)\nRUNTIME_DIR.mkdir(parents=True, exist_ok=True)\n\n# Sampling/window/horizons\nWINDOW_SIZE = 36\nHORIZON_30 = 6\nHORIZON_60 = 12\n\n# BiT backbone (paper-aligned)\nINPUT_DIM = 9\nLSTM_HIDDEN = 64\nN_HEADS = 4\nN_TRANSFORMER_LAYERS = 2\nDROPOUT = 0.2\n\n# MAML\nMETA_EPOCHS = 20\nTASKS_PER_META_BATCH = 32\nINNER_STEPS = 1\nINNER_LR = 5e-4\nMETA_LR = 1e-3\nGRAD_CLIP = 1.0\n\n# Held-out patient adaptation\nFINETUNE_EPOCHS = 30\nFINETUNE_LR = 1e-4\nFINETUNE_BATCH = 64\n\n# Final deployment retrain\nFINAL_EPOCHS = 80\nFINAL_BATCH = 128\nFINAL_LR = 1e-3\n\nMODEL_VERSION = 'bitmaml_1h_v1'\n\n# Keep cleaning flow unchanged, but align features to 9 inputs\nFEATURE_COLS = [\n    'glucose_level',\n    'meal',\n    'bolus',\n    'hypoevent',\n    'sin_hour',\n    'cos_hour',\n    'isnight',\n    'ismealtime',\n    'exercise'\n]\n\nprint('DATASETS_DIR:', DATASETS_DIR, '| exists:', DATASETS_DIR.exists())\nprint('PROCESSED_DIR:', PROCESSED_DIR)\nprint('RUNTIME_DIR:', RUNTIME_DIR)

In [ ]:
# ==== Dataset presence check; fallback fetch in Colab ====\nimport os\nimport io\nimport zipfile\n\ndef _dataset_ready(path: Path) -> bool:\n    return path.exists() and path.is_dir() and any(path.iterdir())\n\ndef fetch_dataset_from_gdrive_if_needed(\n    datasets_dir: Path,\n    file_id='1xtoxijieKG-241NDyscE_qlE6G0UGhl1',\n    zip_name='dataset.zip',\n    extract_to=None\n):\n    if _dataset_ready(datasets_dir):\n        print('Dataset already present:', datasets_dir)\n        return\n\n    if not IN_COLAB:\n        raise FileNotFoundError(f'Dataset missing at {datasets_dir} and auto-fetch is Colab-only.')\n\n    from apiclient import discovery\n    from httplib2 import Http\n    import oauth2client\n    from oauth2client import file, client, tools\n    from googleapiclient.http import MediaIoBaseDownload\n\n    if not Path('client_id.json').exists():\n        raise FileNotFoundError('client_id.json is required in Colab runtime to fetch dataset from Drive API.')\n\n    obj = lambda: None\n    lmao = {"auth_host_name": 'localhost', 'noauth_local_webserver': 'store_true', 'auth_host_port': [8080, 8090], 'logging_level': 'ERROR'}\n    for k, v in lmao.items():\n        setattr(obj, k, v)\n\n    SCOPES = 'https://www.googleapis.com/auth/drive.readonly'\n    store = file.Storage('token.json')\n    creds = store.get()\n    if not creds or creds.invalid:\n        flow = client.flow_from_clientsecrets('client_id.json', SCOPES)\n        creds = tools.run_flow(flow, store, obj)\n\n    DRIVE = discovery.build('drive', 'v3', http=creds.authorize(Http()))\n    request = DRIVE.files().get_media(fileId=file_id)\n\n    fh = io.FileIO(zip_name, mode='wb')\n    downloader = MediaIoBaseDownload(fh, request)\n    done = False\n    while done is False:\n        status, done = downloader.next_chunk()\n        print('Download %d%%.' % int(status.progress() * 100))\n\n    if extract_to is None:\n        extract_to = datasets_dir.parent\n    extract_to = Path(extract_to)\n    extract_to.mkdir(parents=True, exist_ok=True)\n\n    with zipfile.ZipFile(zip_name, 'r') as zf:\n        zf.extractall(extract_to)\n\n    if not _dataset_ready(datasets_dir):\n        raise FileNotFoundError(f'Downloaded and extracted, but dataset path still not ready: {datasets_dir}')\n\n    print('Dataset fetched and ready at:', datasets_dir)\n\nfetch_dataset_from_gdrive_if_needed(DATASETS_DIR)

In [ ]:
# ==== Preprocessing helpers (same cleaning flow retained) ====\ndef is_mealtime(hour: int) -> int:\n    # paper hour set {7,8,12,13,18,19}\n    return int(hour in [7, 8, 12, 13, 18, 19])\n\ndef extract_exercise(note):\n    if pd.isna(note):\n        return 0\n    s = str(note).lower()\n    keys = ['exercise', 'workout', 'run', 'walk', 'gym', 'cycling', 'swim', 'jog']\n    return int(any(k in s for k in keys))\n\ndef create_sliding_windows(df_norm, window_size=36, prediction_horizon=12):\n    values = df_norm[FEATURE_COLS].values.astype(np.float32)\n    glucose = df_norm['glucose_level'].values.astype(np.float32)\n    X, y = [], []\n    max_i = len(df_norm) - window_size - prediction_horizon + 1\n    for i in range(max_i):\n        X.append(values[i:i + window_size])\n        y.append(glucose[i + window_size + prediction_horizon - 1])\n    if not X:\n        return np.empty((0, window_size, len(FEATURE_COLS)), np.float32), np.empty((0,), np.float32)\n    return np.stack(X), np.array(y, np.float32)

In [ ]:
def discover_patient_files(patient_folder: Path):\n    entries_list, treatments_list = [], []\n    for p in patient_folder.rglob('*'):\n        p_low = str(p).lower()\n        if p.is_file() and p.suffix.lower() in ['.csv', '.json_csv'] and 'entries' in p_low:\n            try:\n                df = pd.read_csv(p)\n                if df.shape[1] >= 2:\n                    df = df.iloc[:, :2].copy()\n                    df.columns = ['timestamp', 'sgv']\n                    entries_list.append(df)\n            except Exception:\n                pass\n        elif p.is_file() and p.suffix.lower() == '.json' and 'treatments' in p_low:\n            try:\n                treatments_list.append(pd.read_json(p))\n            except Exception:\n                pass\n    entries_df = pd.concat(entries_list, ignore_index=True) if entries_list else pd.DataFrame()\n    treatments_df = pd.concat(treatments_list, ignore_index=True) if treatments_list else pd.DataFrame()\n    return entries_df, treatments_df\n\ndef process_single_patient(patient_folder: Path, scaler: MinMaxScaler, fit_scaler: bool):\n    pid = patient_folder.name\n    entries_df, treatments_df = discover_patient_files(patient_folder)\n    if entries_df.empty or treatments_df.empty:\n        return None, None, scaler, fit_scaler, pid\n\n    entries_df['timestamp'] = pd.to_datetime(entries_df['timestamp'], utc=True, errors='coerce')\n    entries_df['sgv'] = pd.to_numeric(entries_df['sgv'], errors='coerce')\n    entries_df = entries_df.dropna(subset=['timestamp', 'sgv'])\n    entries_df = entries_df.drop_duplicates(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)\n    if entries_df.empty:\n        return None, None, scaler, fit_scaler, pid\n\n    timeline = pd.date_range(entries_df['timestamp'].min(), entries_df['timestamp'].max(), freq='5min')\n    sgv_df = entries_df.set_index('timestamp').reindex(timeline)\n    sgv_df.index.name = 'timestamp'\n    sgv_df = sgv_df.reset_index().rename(columns={'sgv': 'glucose_level'})\n    sgv_df['glucose_level'] = sgv_df['glucose_level'].interpolate(method='linear', limit=6)\n    sgv_df = sgv_df.dropna(subset=['glucose_level']).reset_index(drop=True)\n\n    if 'created_at' not in treatments_df.columns:\n        return None, None, scaler, fit_scaler, pid\n    treatments_df['timestamp'] = pd.to_datetime(treatments_df['created_at'], utc=True, errors='coerce')\n    treatments_df = treatments_df.dropna(subset=['timestamp']).copy()\n\n    if 'carbs' not in treatments_df.columns:\n        treatments_df['carbs'] = 0\n    if 'insulin' not in treatments_df.columns:\n        treatments_df['insulin'] = 0\n    if 'notes' not in treatments_df.columns:\n        treatments_df['notes'] = ''\n\n    treatments_df['meal'] = pd.to_numeric(treatments_df['carbs'], errors='coerce').fillna(0)\n    treatments_df['bolus'] = pd.to_numeric(treatments_df['insulin'], errors='coerce').fillna(0)\n    treatments_df['exercise'] = treatments_df['notes'].apply(extract_exercise)\n    treatments_df['timestamp_5min'] = treatments_df['timestamp'].dt.round('5min')\n\n    tr_agg = (\n        treatments_df.groupby('timestamp_5min', as_index=False)\n        .agg({'meal': 'sum', 'bolus': 'sum', 'exercise': 'max'})\n        .rename(columns={'timestamp_5min': 'timestamp'})\n    )\n\n    merged = sgv_df.merge(tr_agg, on='timestamp', how='left')\n    merged[['meal', 'bolus', 'exercise']] = merged[['meal', 'bolus', 'exercise']].fillna(0)\n\n    merged['hour'] = merged['timestamp'].dt.hour\n    merged['isnight'] = merged['hour'].apply(lambda h: int(h >= 22 or h <= 5))\n    merged['ismealtime'] = merged['hour'].apply(is_mealtime)\n    merged['hypoevent'] = (merged['glucose_level'] < 70).astype(int)\n    merged['sin_hour'] = np.sin(2 * np.pi * merged['hour'] / 24.0)\n    merged['cos_hour'] = np.cos(2 * np.pi * merged['hour'] / 24.0)\n    merged = merged.drop(columns=['hour'])\n\n    for c in FEATURE_COLS:\n        if c not in merged.columns:\n            merged[c] = 0.0\n\n    merged_norm = merged.copy()\n    if fit_scaler:\n        merged_norm[FEATURE_COLS] = scaler.fit_transform(merged[FEATURE_COLS])\n        fit_scaler = False\n    else:\n        merged_norm[FEATURE_COLS] = scaler.transform(merged[FEATURE_COLS])\n\n    X30, y30 = create_sliding_windows(merged_norm, WINDOW_SIZE, HORIZON_30)\n    X60, y60 = create_sliding_windows(merged_norm, WINDOW_SIZE, HORIZON_60)\n    if len(X30) == 0 or len(X60) == 0:\n        return None, None, scaler, fit_scaler, pid\n\n    return (X30, y30, X60, y60), scaler, fit_scaler, pid

In [ ]:
if not DATASETS_DIR.exists():\n    raise FileNotFoundError(f'Datasets folder not found: {DATASETS_DIR}')\n\npatient_folders = [p for p in DATASETS_DIR.iterdir() if p.is_dir()]\nprint('Patient folders found:', len(patient_folders))\n\nX30_all, y30_all, X60_all, y60_all, ids = [], [], [], [], []\nscaler = MinMaxScaler()\nfit_scaler = True\n\nfor pf in patient_folders:\n    out, scaler, fit_scaler, pid = process_single_patient(pf, scaler, fit_scaler)\n    if out is None:\n        continue\n    X30, y30, X60, y60 = out\n    n = min(len(X30), len(X60))\n    X30_all.append(X30[:n])\n    y30_all.append(y30[:n])\n    X60_all.append(X60[:n])\n    y60_all.append(y60[:n])\n    pid_num = int(re.sub(r'\\D', '', pid) or 0)\n    ids.extend([pid_num] * n)\n\nfinal_X_30min = np.concatenate(X30_all, axis=0).astype(np.float32)\nfinal_y_30min = np.concatenate(y30_all, axis=0).astype(np.float32)\nfinal_X_60min = np.concatenate(X60_all, axis=0).astype(np.float32)\nfinal_y_60min = np.concatenate(y60_all, axis=0).astype(np.float32)\nfinal_patient_ids = np.array(ids, dtype=np.int64)\n\nnp.save(PROCESSED_DIR / 'final_X_30min.npy', final_X_30min)\nnp.save(PROCESSED_DIR / 'final_y_30min.npy', final_y_30min)\nnp.save(PROCESSED_DIR / 'final_X_60min.npy', final_X_60min)\nnp.save(PROCESSED_DIR / 'final_y_60min.npy', final_y_60min)\nnp.save(PROCESSED_DIR / 'final_patient_ids.npy', final_patient_ids)\nwith open(PROCESSED_DIR / 'final_scaler.pkl', 'wb') as f:\n    pickle.dump(scaler, f)\n\nprint('Saved 30m:', final_X_30min.shape, final_y_30min.shape)\nprint('Saved 60m:', final_X_60min.shape, final_y_60min.shape)

In [ ]:
# ==== BiT backbone ====\nclass BiTBackbone(nn.Module):\n    def __init__(self, input_dim=9, lstm_hidden=64, n_heads=4, n_transformer_layers=2, dropout=0.2):\n        super().__init__()\n        self.bilstm = nn.LSTM(input_size=input_dim, hidden_size=lstm_hidden, num_layers=1, batch_first=True, bidirectional=True, dropout=0.0)\n        enc = nn.TransformerEncoderLayer(d_model=2*lstm_hidden, nhead=n_heads, dim_feedforward=4*lstm_hidden, dropout=dropout, batch_first=True, activation='relu')\n        self.transformer = nn.TransformerEncoder(enc, num_layers=n_transformer_layers)\n        self.fc_out = nn.Sequential(\n            nn.Linear(2*lstm_hidden, 64), nn.ReLU(), nn.Dropout(dropout),\n            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),\n            nn.Linear(32, 1)\n        )\n\n    def forward(self, x):\n        lstm_out, _ = self.bilstm(x)\n        trans_out = self.transformer(lstm_out)\n        return self.fc_out(trans_out[:, -1, :])

In [ ]:
# ==== MAML utilities (first-order approximation) ====\ndef mse_loss(model, xb, yb):\n    pred = model(xb)\n    return nn.functional.mse_loss(pred, yb)\n\ndef sample_task_indices(indices, support_size=32, query_size=32):\n    if len(indices) < support_size + query_size:\n        return None, None\n    picked = np.random.choice(indices, size=support_size + query_size, replace=False)\n    return picked[:support_size], picked[support_size:]\n\ndef run_meta_training(X, y, patient_ids, heldout_pid, device):\n    model = BiTBackbone(input_dim=X.shape[2], lstm_hidden=LSTM_HIDDEN, n_heads=N_HEADS, n_transformer_layers=N_TRANSFORMER_LAYERS, dropout=DROPOUT).to(device)\n    meta_opt = torch.optim.Adam(model.parameters(), lr=META_LR)\n\n    train_pids = [pid for pid in np.unique(patient_ids) if pid != heldout_pid]\n    pid_to_idx = {pid: np.where(patient_ids == pid)[0] for pid in train_pids}\n\n    for epoch in range(META_EPOCHS):\n        meta_opt.zero_grad(set_to_none=True)\n        losses = []\n\n        for _ in range(TASKS_PER_META_BATCH):\n            pid = np.random.choice(train_pids)\n            idx = pid_to_idx[pid]\n            s_idx, q_idx = sample_task_indices(idx, support_size=32, query_size=32)\n            if s_idx is None:\n                continue\n\n            support_x = torch.tensor(X[s_idx], dtype=torch.float32, device=device)\n            support_y = torch.tensor(y[s_idx], dtype=torch.float32, device=device).view(-1, 1)\n            query_x = torch.tensor(X[q_idx], dtype=torch.float32, device=device)\n            query_y = torch.tensor(y[q_idx], dtype=torch.float32, device=device).view(-1, 1)\n\n            fast_model = copy.deepcopy(model)\n            fast_opt = torch.optim.SGD(fast_model.parameters(), lr=INNER_LR)\n\n            for _ in range(INNER_STEPS):\n                fast_opt.zero_grad(set_to_none=True)\n                s_loss = mse_loss(fast_model, support_x, support_y)\n                s_loss.backward()\n                torch.nn.utils.clip_grad_norm_(fast_model.parameters(), GRAD_CLIP)\n                fast_opt.step()\n\n            q_loss = mse_loss(fast_model, query_x, query_y)\n            losses.append(q_loss)\n\n        if not losses:\n            continue\n\n        meta_loss = torch.stack(losses).mean()\n        meta_loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)\n        meta_opt.step()\n\n        if epoch % 5 == 0:\n            print(f'[meta] epoch={epoch} loss={meta_loss.item():.6f}')\n\n    return model.state_dict()

In [ ]:
# ==== LOPO-CV with heldout 70/30 adaptation split ====\ndef evaluate_lopo_bitmaml(X, y, patient_ids, horizon_label='60m'):\n    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n    uniq = np.unique(patient_ids)\n\n    with open(PROCESSED_DIR / 'final_scaler.pkl', 'rb') as f:\n        scaler = pickle.load(f)\n\n    all_pred_norm, all_true_norm = [], []\n\n    for fold, pid in enumerate(uniq, start=1):\n        print(f'\n[LOPO {horizon_label}] fold {fold}/{len(uniq)} heldout pid={pid}')\n\n        meta_state = run_meta_training(X, y, patient_ids, heldout_pid=pid, device=device)\n\n        held_idx = np.where(patient_ids == pid)[0]\n        X_held = X[held_idx]\n        y_held = y[held_idx]\n\n        split = int(0.7 * len(X_held))\n        X_ft, y_ft = X_held[:split], y_held[:split]\n        X_te, y_te = X_held[split:], y_held[split:]\n        if len(X_ft) == 0 or len(X_te) == 0:\n            print('skip fold due to insufficient heldout samples')\n            continue\n\n        model = BiTBackbone(input_dim=X.shape[2], lstm_hidden=LSTM_HIDDEN, n_heads=N_HEADS, n_transformer_layers=N_TRANSFORMER_LAYERS, dropout=DROPOUT).to(device)\n        model.load_state_dict(meta_state)\n\n        ft_loader = DataLoader(TensorDataset(torch.tensor(X_ft, dtype=torch.float32), torch.tensor(y_ft, dtype=torch.float32).view(-1, 1)), batch_size=FINETUNE_BATCH, shuffle=True)\n        opt = torch.optim.Adam(model.parameters(), lr=FINETUNE_LR)\n        scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5)\n        crit = nn.MSELoss()\n\n        for _ in range(FINETUNE_EPOCHS):\n            model.train()\n            for xb, yb in ft_loader:\n                xb, yb = xb.to(device), yb.to(device)\n                opt.zero_grad(set_to_none=True)\n                loss = crit(model(xb), yb)\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)\n                opt.step()\n            scheduler.step()\n\n        model.eval()\n        with torch.no_grad():\n            pred_norm = model(torch.tensor(X_te, dtype=torch.float32).to(device)).cpu().numpy().reshape(-1)\n\n        all_pred_norm.append(pred_norm)\n        all_true_norm.append(y_te.reshape(-1))\n\n    pred_norm = np.concatenate(all_pred_norm)\n    true_norm = np.concatenate(all_true_norm)\n\n    z = np.zeros((len(pred_norm), len(FEATURE_COLS)-1), dtype=np.float32)\n    pred_mgdl = scaler.inverse_transform(np.hstack([pred_norm.reshape(-1, 1), z]))[:, 0]\n    true_mgdl = scaler.inverse_transform(np.hstack([true_norm.reshape(-1, 1), z]))[:, 0]\n\n    rmse = np.sqrt(mean_squared_error(true_mgdl, pred_mgdl))\n    mae = mean_absolute_error(true_mgdl, pred_mgdl)\n    print(f'[LOPO {horizon_label}] RMSE={rmse:.2f} mg/dL MAE={mae:.2f} mg/dL')\n    return rmse, mae

In [ ]:
X30 = np.load(PROCESSED_DIR / 'final_X_30min.npy')\ny30 = np.load(PROCESSED_DIR / 'final_y_30min.npy')\nX60 = np.load(PROCESSED_DIR / 'final_X_60min.npy')\ny60 = np.load(PROCESSED_DIR / 'final_y_60min.npy')\npid = np.load(PROCESSED_DIR / 'final_patient_ids.npy')\n\nrmse30, mae30 = evaluate_lopo_bitmaml(X30, y30, pid, horizon_label='30m')\nrmse60, mae60 = evaluate_lopo_bitmaml(X60, y60, pid, horizon_label='60m')

In [ ]:
# ==== Final 60-min deployment model + export ====\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nX = np.load(PROCESSED_DIR / 'final_X_60min.npy')\ny = np.load(PROCESSED_DIR / 'final_y_60min.npy')\n\nmodel = BiTBackbone(input_dim=X.shape[2], lstm_hidden=LSTM_HIDDEN, n_heads=N_HEADS, n_transformer_layers=N_TRANSFORMER_LAYERS, dropout=DROPOUT).to(device)\nopt = torch.optim.Adam(model.parameters(), lr=FINAL_LR)\ncrit = nn.MSELoss()\nloader = DataLoader(TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32).view(-1, 1)), batch_size=FINAL_BATCH, shuffle=True)\n\nfor epoch in range(FINAL_EPOCHS):\n    model.train()\n    total = 0.0\n    for xb, yb in loader:\n        xb, yb = xb.to(device), yb.to(device)\n        opt.zero_grad(set_to_none=True)\n        loss = crit(model(xb), yb)\n        loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)\n        opt.step()\n        total += loss.item()\n    if epoch % 10 == 0:\n        print(f'[final] epoch={epoch} loss={total/max(1,len(loader)):.6f}')\n\nbest_path = PROCESSED_DIR / 'best_bitmaml.pth'\ntorch.save(model.state_dict(), best_path)\n\nexport_model = BiTBackbone(input_dim=X.shape[2], lstm_hidden=LSTM_HIDDEN, n_heads=N_HEADS, n_transformer_layers=N_TRANSFORMER_LAYERS, dropout=DROPOUT)\nexport_model.load_state_dict(torch.load(best_path, map_location='cpu'))\nexport_model.eval()\n\nonnx_path = RUNTIME_DIR / 'model.onnx'\ndummy = torch.randn(1, X.shape[1], X.shape[2], dtype=torch.float32)\ntorch.onnx.export(\n    export_model, dummy, str(onnx_path),\n    input_names=['input'], output_names=['pred'],\n    dynamic_axes={'input': {0: 'batch'}, 'pred': {0: 'batch'}},\n    opset_version=18, do_constant_folding=True, dynamo=False\n)\nonnx.checker.check_model(onnx.load(str(onnx_path)))\n\nwith open(PROCESSED_DIR / 'final_scaler.pkl', 'rb') as f:\n    scaler = pickle.load(f)\n\nmetadata = {\n    'model_version': MODEL_VERSION,\n    'model_file': 'model.onnx',\n    'window_size': int(X.shape[1]),\n    'horizon_steps': HORIZON_60,\n    'feature_order': FEATURE_COLS,\n    'output': {'name': 'predicted_sgv_norm', 'unit_after_denorm': 'mg/dL'},\n    'scaler': {\n        'type': 'minmax',\n        'data_min': scaler.data_min_.tolist(),\n        'data_max': scaler.data_max_.tolist(),\n        'scale': scaler.scale_.tolist(),\n        'min_offset': scaler.min_.tolist()\n    }\n}\n\nwith open(RUNTIME_DIR / 'metadata.json', 'w') as f:\n    json.dump(metadata, f, indent=2)\n\nprint('Exported:', onnx_path)\nprint('Exported:', RUNTIME_DIR / 'metadata.json')

In [ ]:
# Copy artifacts into Android app assets\n# Update path if needed\nANDROID_ASSETS_ML = Path('/content/T1DAlert2/app/src/main/assets/ml')\nANDROID_ASSETS_ML.mkdir(parents=True, exist_ok=True)\n\nimport shutil\nshutil.copy2(RUNTIME_DIR / 'model.onnx', ANDROID_ASSETS_ML / 'model.onnx')\nshutil.copy2(RUNTIME_DIR / 'metadata.json', ANDROID_ASSETS_ML / 'metadata.json')\nprint('Copied runtime artifacts to:', ANDROID_ASSETS_ML)